In [1]:
from pathlib import Path
import sys
import time
import math
import numpy as np
import pandas as pd
from IPython.display import display

from scipy.sparse import csr_matrix
from scipy.optimize import milp, LinearConstraint, Bounds

sys.path.append('курсовая_2')

from qubo_cycle import build_var_map, build_cycle_qubo
from ising_mapping import qubo_to_ising
from pair_search import verify_single_dummy_cycle, extract_pair_and_weight
from SB_modules import BSB, DSB

In [ ]:
# 04_05_03: REAL MARKET SENSITIVITY EXPERIMENTS
# Threshold sensitivity, SB cost-scale sensitivity, penalty sensitivity

# 1. CONFIG

TAG = "04_05_03"

SAVE_DIR = Path("04_05") if Path("04_05").exists() else Path(".")
SAVE_DIR.mkdir(exist_ok=True)

THRESHOLDS = [-0.005, -0.01, -0.015, -0.02, -0.03, -0.04, -0.05, -0.055]

MC = 1.0
MP_BASE = 1.0
DT = 0.5
N_ITER = 500
SB_VARIANT = "BSB"

# Sensitivity runtime parameters
MAX_PAIRS = 10
N_RUNS_SB = 100
SA_RESTARTS = 30
SA_STEPS = 3000

BASE_SEED = 450503

RUN_THRESHOLD_SENSITIVITY = True
RUN_SB_SCALE_SENSITIVITY = True
RUN_SB_PENALTY_SENSITIVITY = True


def locate_file(filename: str) -> Path:
    candidates = [
        Path(filename),
        Path("04_05") / filename,
        Path("../04_05") / filename,
        Path("/mnt/data") / filename,
        Path("/mnt/data/04_05") / filename,
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("Cannot find file:\n" + "\n".join(str(p) for p in candidates))


weights_path = locate_file("real_data_market_weights_04_05_01.csv")

w_df = pd.read_csv(weights_path, index_col=0)
symbols_with_dummy = list(w_df.index)
w_raw = w_df.to_numpy(dtype=float)

N = w_raw.shape[0] - 1
node_to_symbol = {i: s for i, s in enumerate(symbols_with_dummy)}

stock_block = w_raw[1:, 1:]
off_diag_mask = ~np.eye(N, dtype=bool)
max_abs_stock_weight = float(np.max(np.abs(stock_block[off_diag_mask])))

COST_SCALE_REF = 1.0 if max_abs_stock_weight <= 1e-15 else 1.0 / max_abs_stock_weight

COST_SCALES = [1.0, 10.0, 25.0, COST_SCALE_REF, 50.0, 100.0]
PENALTIES = [0.5, 1.0, 2.0, 5.0, 10.0]

print("SAVE_DIR:", SAVE_DIR.resolve())
print("weights:", weights_path)
print("N stock nodes:", N)
print("max_abs_stock_weight:", max_abs_stock_weight)
print("COST_SCALE_REF:", COST_SCALE_REF)


# 2. COMMON HELPERS

def path_weight(path_nodes, w):
    return float(sum(w[a, b] for a, b in zip(path_nodes[:-1], path_nodes[1:])))


def path_symbols(path_nodes):
    return "->".join(node_to_symbol.get(i, str(i)) for i in path_nodes)


def make_tabu_matrix(forbidden_pairs):
    T = np.zeros_like(w_raw, dtype=int)
    for short, long in forbidden_pairs:
        T[long, short] = 1
    return T


def _extract_runtime_sec(df) -> float:
    if df is None or len(df) == 0:
        return np.nan

    if "total_mode_runtime_sec" in df.columns:
        vals = pd.to_numeric(df["total_mode_runtime_sec"], errors="coerce").dropna()
        if len(vals):
            return float(vals.iloc[0])

    if "runtime_sec" in df.columns:
        vals = pd.to_numeric(df["runtime_sec"], errors="coerce").dropna()
        if len(vals):
            return float(vals.sum())

    return np.nan


def summarize_pairs(df, solver, threshold, extra=None):
    if df is None or len(df) == 0:
        out = {
            "solver": solver,
            "threshold": threshold,
            "n_accepted_pairs": 0,
            "best_path_weight": np.nan,
            "mean_path_weight": np.nan,
            "median_path_weight": np.nan,
            "n_bypass": 0,
            "bypass_share": np.nan,
            "duplicate_endpoint_pairs": 0,
            "mean_path_length_assets": np.nan,
            "max_path_length_assets": np.nan,
            "runtime_sec": np.nan,
        }
    else:
        accepted = df[df["accepted"] == True].copy()
        if "path_weight" in accepted.columns:
            accepted = accepted[accepted["path_weight"] < threshold].copy()

        if len(accepted):
            endpoint_counts = (
                accepted.groupby(["short_node", "long_node"], dropna=False)
                .size()
            )
            duplicate_endpoint_pairs = int((endpoint_counts > 1).sum())
            n_bypass = int(accepted["is_bypass"].fillna(False).astype(bool).sum())

            if "path_length_assets" in accepted.columns:
                mean_path_length_assets = float(accepted["path_length_assets"].mean())
                max_path_length_assets = int(accepted["path_length_assets"].max())
            else:
                mean_path_length_assets = np.nan
                max_path_length_assets = np.nan
        else:
            duplicate_endpoint_pairs = 0
            n_bypass = 0
            mean_path_length_assets = np.nan
            max_path_length_assets = np.nan

        out = {
            "solver": solver,
            "threshold": threshold,
            "n_accepted_pairs": int(len(accepted)),
            "best_path_weight": float(accepted["path_weight"].min()) if len(accepted) else np.nan,
            "mean_path_weight": float(accepted["path_weight"].mean()) if len(accepted) else np.nan,
            "median_path_weight": float(accepted["path_weight"].median()) if len(accepted) else np.nan,
            "n_bypass": n_bypass,
            "bypass_share": float(n_bypass / len(accepted)) if len(accepted) else np.nan,
            "duplicate_endpoint_pairs": duplicate_endpoint_pairs,
            "mean_path_length_assets": mean_path_length_assets,
            "max_path_length_assets": max_path_length_assets,
            "runtime_sec": _extract_runtime_sec(df),
        }

    if extra:
        out.update(extra)

    return out

# 3. MILP / HiGHS EXACT SOLVER

var_map = build_var_map(N)
edges = var_map.edges
M = len(edges)

edge_to_u = var_map.e2u
u_to_edge = var_map.u2e


def detect_stock_subcycles(selected_edges):
    """
    Returns stock-only directed cycles not containing dummy node 0.
    """
    out = {}
    for i, j in selected_edges:
        if i > 0 and j > 0:
            out[i] = j

    cycles = []
    globally_seen = set()

    for start in range(1, N + 1):
        if start in globally_seen:
            continue

        local_pos = {}
        cur = start
        path = []

        while cur in out:
            if cur in local_pos:
                cyc = path[local_pos[cur]:]
                if len(cyc) >= 2:
                    cycles.append(cyc)
                break

            if cur in globally_seen:
                break

            local_pos[cur] = len(path)
            path.append(cur)
            cur = out[cur]

        globally_seen.update(path)

    return cycles


def build_milp_constraints(forbidden_pairs=None, sec_cycles=None):
    if forbidden_pairs is None:
        forbidden_pairs = set()
    if sec_cycles is None:
        sec_cycles = []

    rows = []
    cols = []
    data = []
    lb = []
    ub = []
    row_id = 0

    def add_constraint(coeffs, lo, hi):
        nonlocal row_id
        for u, val in coeffs.items():
            if abs(val) > 1e-15:
                rows.append(row_id)
                cols.append(u)
                data.append(float(val))
        lb.append(lo)
        ub.append(hi)
        row_id += 1

    # dummy out-degree = 1
    add_constraint(
        {edge_to_u[(0, j)]: 1.0 for j in range(1, N + 1)},
        1.0,
        1.0,
    )

    # dummy in-degree = 1
    add_constraint(
        {edge_to_u[(i, 0)]: 1.0 for i in range(1, N + 1)},
        1.0,
        1.0,
    )

    # stock in/out degree <= 1
    for i in range(1, N + 1):
        add_constraint(
            {edge_to_u[(i, j)]: 1.0 for j in range(0, N + 1) if j != i},
            0.0,
            1.0,
        )
        add_constraint(
            {edge_to_u[(j, i)]: 1.0 for j in range(0, N + 1) if j != i},
            0.0,
            1.0,
        )

    # flow balance out_i - in_i = 0 for all nodes
    for i in range(0, N + 1):
        coeffs = {}
        for j in range(0, N + 1):
            if j == i:
                continue
            coeffs[edge_to_u[(i, j)]] = coeffs.get(edge_to_u[(i, j)], 0.0) + 1.0
            coeffs[edge_to_u[(j, i)]] = coeffs.get(edge_to_u[(j, i)], 0.0) - 1.0
        add_constraint(coeffs, 0.0, 0.0)

    # forbid trivial 0 -> a -> 0 by requiring at least one stock-stock edge
    add_constraint(
        {
            edge_to_u[(i, j)]: 1.0
            for i in range(1, N + 1)
            for j in range(1, N + 1)
            if i != j
        },
        1.0,
        np.inf,
    )

    # anti-parallel edges: x_ij + x_ji <= 1
    for i in range(0, N + 1):
        for j in range(i + 1, N + 1):
            add_constraint(
                {
                    edge_to_u[(i, j)]: 1.0,
                    edge_to_u[(j, i)]: 1.0,
                },
                0.0,
                1.0,
            )

    # endpoint tabu: forbid choosing both 0->short and long->0
    for short, long in forbidden_pairs:
        add_constraint(
            {
                edge_to_u[(0, short)]: 1.0,
                edge_to_u[(long, 0)]: 1.0,
            },
            0.0,
            1.0,
        )

    # subtour elimination constraints for detected stock-only cycles
    for cyc in sec_cycles:
        S = list(cyc)
        coeffs = {}
        for i in S:
            for j in S:
                if i != j:
                    coeffs[edge_to_u[(i, j)]] = 1.0
        add_constraint(coeffs, 0.0, len(S) - 1)

    A = csr_matrix((data, (rows, cols)), shape=(row_id, M))
    return LinearConstraint(A, np.array(lb, dtype=float), np.array(ub, dtype=float))


def solve_milp_one(w, forbidden_pairs=None, time_limit=30.0):
    if forbidden_pairs is None:
        forbidden_pairs = set()

    c = np.array([w[i, j] for i, j in edges], dtype=float)

    sec_cycles = []
    total_start = time.perf_counter()

    for sec_iter in range(30):
        constraints = build_milp_constraints(
            forbidden_pairs=forbidden_pairs,
            sec_cycles=sec_cycles,
        )

        res = milp(
            c=c,
            integrality=np.ones(M),
            bounds=Bounds(0.0, 1.0),
            constraints=constraints,
            options={
                "time_limit": time_limit,
                "mip_rel_gap": 0.0,
                "disp": False,
            },
        )

        runtime = time.perf_counter() - total_start

        if not res.success or res.x is None:
            return {
                "ok": False,
                "status": "milp_failed",
                "message": str(res.message),
                "runtime_sec": runtime,
            }

        z = (res.x >= 0.5).astype(int)
        selected_edges = [u_to_edge[u] for u, bit in enumerate(z) if bit == 1]

        ok, cycle = verify_single_dummy_cycle(selected_edges, N=N)
        if ok:
            short, long, path_nodes, pw = extract_pair_and_weight(cycle, w_raw)
            return {
                "ok": True,
                "status": "ok",
                "short_node": short,
                "long_node": long,
                "short_symbol": node_to_symbol[short],
                "long_symbol": node_to_symbol[long],
                "path_nodes": "->".join(map(str, path_nodes)),
                "path_symbols": path_symbols(path_nodes),
                "path_length_assets": len(path_nodes),
                "path_weight": float(pw),
                "direct_weight": float(w_raw[short, long]),
                "is_bypass": bool(len(path_nodes) > 2),
                "runtime_sec": runtime,
                "objective": float(res.fun),
                "sec_iterations": sec_iter,
            }

        new_cycles = detect_stock_subcycles(selected_edges)
        if not new_cycles:
            return {
                "ok": False,
                "status": "invalid_no_detected_subcycle",
                "runtime_sec": runtime,
                "selected_edges": str(selected_edges),
            }

        sec_cycles.extend(new_cycles)

    return {
        "ok": False,
        "status": "too_many_sec_iterations",
        "runtime_sec": time.perf_counter() - total_start,
    }


def run_milp_sequential(threshold, max_pairs=MAX_PAIRS):
    forbidden = set()
    rows = []

    for it in range(max_pairs):
        sol = solve_milp_one(w_raw, forbidden_pairs=forbidden)

        if not sol["ok"]:
            rows.append({
                "solver": "MILP_HiGHS_exact",
                "iteration": it,
                "accepted": False,
                "status": sol["status"],
                "threshold": threshold,
                "runtime_sec": sol.get("runtime_sec", np.nan),
            })
            break

        accepted = sol["path_weight"] < threshold
        row = {
            "solver": "MILP_HiGHS_exact",
            "iteration": it,
            "accepted": bool(accepted),
            "status": "ok" if accepted else "stopped_by_threshold",
            "threshold": threshold,
            **sol,
        }
        rows.append(row)

        if not accepted:
            break

        forbidden.add((sol["short_node"], sol["long_node"]))

    return pd.DataFrame(rows)


# 4. SIMULATED ANNEALING VALID-PATH BASELINE

def random_valid_path(rng, forbidden_pairs, max_len=None):
    if max_len is None:
        max_len = N
    nodes = list(range(1, N + 1))

    for _ in range(1000):
        L = int(rng.integers(2, max_len + 1))
        path = list(rng.choice(nodes, size=L, replace=False))
        if (path[0], path[-1]) not in forbidden_pairs:
            return path

    # Fallback
    for a in nodes:
        for b in nodes:
            if a != b and (a, b) not in forbidden_pairs:
                return [a, b]

    return None


def propose_neighbor(path, rng, forbidden_pairs):
    path = list(path)
    nodes_all = set(range(1, N + 1))
    move = rng.choice(["replace", "insert", "remove", "swap", "reverse"])

    for _ in range(100):
        q = list(path)

        if move == "replace":
            pos = int(rng.integers(0, len(q)))
            available = list(nodes_all - set(q))
            if not available:
                continue
            q[pos] = int(rng.choice(available))

        elif move == "insert":
            if len(q) >= N:
                continue
            available = list(nodes_all - set(q))
            if not available:
                continue
            pos = int(rng.integers(0, len(q) + 1))
            q.insert(pos, int(rng.choice(available)))

        elif move == "remove":
            if len(q) <= 2:
                continue
            pos = int(rng.integers(0, len(q)))
            q.pop(pos)

        elif move == "swap":
            if len(q) < 2:
                continue
            a, b = rng.choice(len(q), size=2, replace=False)
            q[int(a)], q[int(b)] = q[int(b)], q[int(a)]

        elif move == "reverse":
            if len(q) < 3:
                continue
            a, b = sorted(rng.choice(len(q), size=2, replace=False))
            q[int(a):int(b)+1] = reversed(q[int(a):int(b)+1])

        if len(q) >= 2 and len(set(q)) == len(q) and q[0] != q[-1]:
            if (q[0], q[-1]) not in forbidden_pairs:
                return q

    return path


def solve_sa_one(threshold, forbidden_pairs, seed):
    rng = np.random.default_rng(seed)
    best_path = None
    best_E = np.inf
    start = time.perf_counter()

    for restart in range(SA_RESTARTS):
        cur = random_valid_path(rng, forbidden_pairs)
        if cur is None:
            break

        cur_E = path_weight(cur, w_raw)
        T0 = max(1e-3, abs(cur_E) + max_abs_stock_weight)
        Tmin = 1e-5

        for step in range(SA_STEPS):
            frac = step / max(1, SA_STEPS - 1)
            T = T0 * ((Tmin / T0) ** frac)

            cand = propose_neighbor(cur, rng, forbidden_pairs)
            cand_E = path_weight(cand, w_raw)

            dE = cand_E - cur_E
            if dE <= 0 or rng.random() < math.exp(-dE / max(T, 1e-12)):
                cur = cand
                cur_E = cand_E

            if cur_E < best_E:
                best_E = cur_E
                best_path = list(cur)

    runtime = time.perf_counter() - start

    if best_path is None:
        return {
            "ok": False,
            "status": "sa_failed",
            "runtime_sec": runtime,
        }

    short = best_path[0]
    long = best_path[-1]

    return {
        "ok": True,
        "status": "ok",
        "short_node": short,
        "long_node": long,
        "short_symbol": node_to_symbol[short],
        "long_symbol": node_to_symbol[long],
        "path_nodes": "->".join(map(str, best_path)),
        "path_symbols": path_symbols(best_path),
        "path_length_assets": len(best_path),
        "path_weight": float(best_E),
        "direct_weight": float(w_raw[short, long]),
        "is_bypass": bool(len(best_path) > 2),
        "runtime_sec": runtime,
    }


def run_sa_sequential(threshold, max_pairs=MAX_PAIRS, seed=BASE_SEED):
    forbidden = set()
    rows = []

    for it in range(max_pairs):
        sol = solve_sa_one(
            threshold=threshold,
            forbidden_pairs=forbidden,
            seed=seed + 10000 * it,
        )

        if not sol["ok"]:
            rows.append({
                "solver": "SA_valid_path",
                "iteration": it,
                "accepted": False,
                "status": sol["status"],
                "threshold": threshold,
                "runtime_sec": sol.get("runtime_sec", np.nan),
            })
            break

        accepted = sol["path_weight"] < threshold

        rows.append({
            "solver": "SA_valid_path",
            "iteration": it,
            "accepted": bool(accepted),
            "status": "ok" if accepted else "stopped_by_threshold",
            "threshold": threshold,
            **sol,
        })

        if not accepted:
            break

        forbidden.add((sol["short_node"], sol["long_node"]))

    return pd.DataFrame(rows)


# 5. SB SEQUENTIAL SENSITIVITY

def decode_edges(z, var_map):
    z = np.asarray(z, dtype=int).reshape(-1)
    return [var_map.u2e[u] for u, bit in enumerate(z) if bit == 1]


def run_sb_once(J, h, seed):
    if SB_VARIANT.upper() == "BSB":
        solver = BSB(J, h=h, n_iter=N_ITER, dt=DT, seed=seed)
    elif SB_VARIANT.upper() == "DSB":
        solver = DSB(J, h=h, n_iter=N_ITER, dt=DT, seed=seed)
    else:
        raise ValueError("SB_VARIANT must be BSB or DSB")

    solver.run(record_trajectory=False)

    x = solver.x.reshape(-1)
    s = np.where(x >= 0.0, 1, -1)
    z = ((s + 1) // 2).astype(int)
    return z


def run_sb_sequential(
    threshold,
    cost_scale,
    mp,
    mode_name,
    max_pairs=MAX_PAIRS,
    n_runs=N_RUNS_SB,
    seed_offset=0,
):
    w_for_qubo = w_raw * float(cost_scale)

    forbidden = set()
    pair_rows = []
    iter_rows = []

    total_start = time.perf_counter()

    for it in range(max_pairs):
        T = make_tabu_matrix(forbidden)

        lin, quad, vm = build_cycle_qubo(
            w=w_for_qubo,
            tabu=T,
            mc=MC,
            mp=mp,
        )
        J, h, _ = qubo_to_ising(lin, quad)

        run_candidates = []

        iter_start = time.perf_counter()

        for r in range(n_runs):
            seed = BASE_SEED + seed_offset + 100000 * it + r
            z = run_sb_once(J, h, seed=seed)

            selected_edges = decode_edges(z, vm)
            ok, cycle = verify_single_dummy_cycle(selected_edges, N=N)

            row = {
                "mode": mode_name,
                "iteration": it,
                "run": r,
                "threshold": threshold,
                "cost_scale": cost_scale,
                "mp": mp,
                "n_tabu_before": len(forbidden),
                "n_selected_edges": len(selected_edges),
                "valid_cycle": bool(ok),
                "endpoint_in_tabu_before": False,
                "below_threshold": False,
                "eligible_after_hard_filter": False,
                "path_weight_raw": np.nan,
                "path_weight_qubo": np.nan,
                "short_node": np.nan,
                "long_node": np.nan,
                "path_nodes": np.nan,
                "path_symbols": np.nan,
                "path_length_assets": 0,
            }

            if ok:
                short, long, path_nodes, pw_raw = extract_pair_and_weight(cycle, w_raw)
                _, _, _, pw_qubo = extract_pair_and_weight(cycle, w_for_qubo)

                endpoint_in_tabu = bool(T[long, short] == 1)
                below = bool(pw_raw < threshold)
                eligible = bool((not endpoint_in_tabu) and below and short != long)

                row.update({
                    "short_node": int(short),
                    "long_node": int(long),
                    "short_symbol": node_to_symbol[short],
                    "long_symbol": node_to_symbol[long],
                    "path_nodes": "->".join(map(str, path_nodes)),
                    "path_symbols": path_symbols(path_nodes),
                    "path_length_assets": len(path_nodes),
                    "path_weight_raw": float(pw_raw),
                    "path_weight_qubo": float(pw_qubo),
                    "direct_weight_raw": float(w_raw[short, long]),
                    "endpoint_in_tabu_before": endpoint_in_tabu,
                    "below_threshold": below,
                    "eligible_after_hard_filter": eligible,
                })

            run_candidates.append(row)

        cand_df = pd.DataFrame(run_candidates)

        valid_cycle_rate = float(cand_df["valid_cycle"].mean())
        nonzero_edge_rate = float((cand_df["n_selected_edges"] > 0).mean())

        valid_df = cand_df[cand_df["valid_cycle"] == True]
        n_tabu_viol = int(valid_df["endpoint_in_tabu_before"].sum()) if len(valid_df) else 0
        tabu_viol_rate_valid = float(n_tabu_viol / len(valid_df)) if len(valid_df) else np.nan

        eligible_df = cand_df[cand_df["eligible_after_hard_filter"] == True].copy()

        iter_rows.append({
            "mode": mode_name,
            "iteration": it,
            "threshold": threshold,
            "cost_scale": cost_scale,
            "mp": mp,
            "n_tabu_before": len(forbidden),
            "valid_cycle_rate": valid_cycle_rate,
            "nonzero_edge_rate": nonzero_edge_rate,
            "n_valid_cycles": int(cand_df["valid_cycle"].sum()),
            "n_tabu_violations_among_valid": n_tabu_viol,
            "tabu_violation_rate_among_valid": tabu_viol_rate_valid,
            "n_eligible_after_hard_filter": int(len(eligible_df)),
            "iteration_runtime_sec": time.perf_counter() - iter_start,
        })

        if eligible_df.empty:
            break

        best = eligible_df.sort_values(
            ["path_weight_raw", "run"],
            ascending=[True, True],
        ).iloc[0]

        short = int(best["short_node"])
        long = int(best["long_node"])
        path_len = int(best["path_length_assets"])

        pair_rows.append({
            "solver": mode_name,
            "mode": mode_name,
            "iteration": it,
            "accepted": True,
            "status": "ok",
            "threshold": threshold,
            "cost_scale": cost_scale,
            "mp": mp,
            "short_node": short,
            "long_node": long,
            "short_symbol": best["short_symbol"],
            "long_symbol": best["long_symbol"],
            "path_nodes": best["path_nodes"],
            "path_symbols": best["path_symbols"],
            "path_length_assets": path_len,
            "path_weight": float(best["path_weight_raw"]),
            "path_weight_qubo": float(best["path_weight_qubo"]),
            "direct_weight": float(best["direct_weight_raw"]),
            "is_bypass": bool(path_len > 2),
            "runtime_sec": np.nan,
            "chosen_run": int(best["run"]),
        })

        forbidden.add((short, long))

    pairs_df = pd.DataFrame(pair_rows)
    iter_df = pd.DataFrame(iter_rows)

    total_runtime = time.perf_counter() - total_start
    if len(pairs_df):
        pairs_df["total_mode_runtime_sec"] = total_runtime

    return pairs_df, iter_df


# 6. THRESHOLD SENSITIVITY

threshold_rows = []
threshold_pairs_frames = []
threshold_sb_iter_frames = []

if RUN_THRESHOLD_SENSITIVITY:
    for ti, th in enumerate(THRESHOLDS):
        print("\nThreshold sensitivity:", th)

        # MILP
        milp_df = run_milp_sequential(threshold=th)
        threshold_pairs_frames.append(milp_df)
        threshold_rows.append(summarize_pairs(
            milp_df,
            solver="MILP_HiGHS_exact",
            threshold=th,
            extra={"experiment": "threshold_sensitivity"},
        ))

        # SA
        sa_df = run_sa_sequential(threshold=th, seed=BASE_SEED + 1000 * ti)
        threshold_pairs_frames.append(sa_df)
        threshold_rows.append(summarize_pairs(
            sa_df,
            solver="SA_valid_path",
            threshold=th,
            extra={"experiment": "threshold_sensitivity"},
        ))

        # SB scaled diagnostic with fixed mp
        sb_pairs, sb_iters = run_sb_sequential(
            threshold=th,
            cost_scale=COST_SCALE_REF,
            mp=1.0,
            mode_name="SB_scaled_fixed_mp",
            seed_offset=2000000 + 10000 * ti,
        )

        threshold_pairs_frames.append(sb_pairs)
        threshold_sb_iter_frames.append(sb_iters)

        sb_extra = {
            "experiment": "threshold_sensitivity",
            "cost_scale": COST_SCALE_REF,
            "mp": 1.0,
        }

        if len(sb_iters):
            sb_extra.update({
                "runtime_sec": float(sb_iters["iteration_runtime_sec"].sum()),
                "mean_valid_cycle_rate": float(sb_iters["valid_cycle_rate"].mean()),
                "total_tabu_violations": int(sb_iters["n_tabu_violations_among_valid"].sum()),
                "mean_tabu_violation_rate_among_valid": float(sb_iters["tabu_violation_rate_among_valid"].mean(skipna=True)),
            })

        threshold_rows.append(summarize_pairs(
            sb_pairs,
            solver="SB_scaled_fixed_mp",
            threshold=th,
            extra=sb_extra,
        ))

threshold_sensitivity = pd.DataFrame(threshold_rows)
threshold_pairs = pd.concat(threshold_pairs_frames, ignore_index=True, sort=False) if threshold_pairs_frames else pd.DataFrame()
threshold_sb_iterations = pd.concat(threshold_sb_iter_frames, ignore_index=True, sort=False) if threshold_sb_iter_frames else pd.DataFrame()


# 7. SB COST-SCALE SENSITIVITY

scale_rows = []
scale_pairs_frames = []
scale_iter_frames = []

if RUN_SB_SCALE_SENSITIVITY:
    fixed_threshold = -0.01

    for si, cs in enumerate(COST_SCALES):
        print("\nSB cost-scale sensitivity:", cs)

        mode_name = f"SB_scale_{cs:.6g}"

        sb_pairs, sb_iters = run_sb_sequential(
            threshold=fixed_threshold,
            cost_scale=cs,
            mp=1.0,
            mode_name=mode_name,
            seed_offset=3000000 + 10000 * si,
        )

        scale_pairs_frames.append(sb_pairs)
        scale_iter_frames.append(sb_iters)

        extra = {
            "experiment": "sb_cost_scale_sensitivity",
            "cost_scale": cs,
            "mp": 1.0,
        }

        if len(sb_iters):
            extra.update({
                "runtime_sec": float(sb_iters["iteration_runtime_sec"].sum()),
                "mean_valid_cycle_rate": float(sb_iters["valid_cycle_rate"].mean()),
                "total_tabu_violations": int(sb_iters["n_tabu_violations_among_valid"].sum()),
                "mean_tabu_violation_rate_among_valid": float(sb_iters["tabu_violation_rate_among_valid"].mean(skipna=True)),
            })

        scale_rows.append(summarize_pairs(
            sb_pairs,
            solver=mode_name,
            threshold=fixed_threshold,
            extra=extra,
        ))

scale_sensitivity = pd.DataFrame(scale_rows)
scale_pairs = pd.concat(scale_pairs_frames, ignore_index=True, sort=False) if scale_pairs_frames else pd.DataFrame()
scale_iterations = pd.concat(scale_iter_frames, ignore_index=True, sort=False) if scale_iter_frames else pd.DataFrame()


# 8. SB PENALTY SENSITIVITY

penalty_rows = []
penalty_pairs_frames = []
penalty_iter_frames = []

if RUN_SB_PENALTY_SENSITIVITY:
    fixed_threshold = -0.01

    for pi, mp in enumerate(PENALTIES):
        print("\nSB penalty sensitivity:", mp)

        mode_name = f"SB_scaled_mp_{mp:.6g}"

        sb_pairs, sb_iters = run_sb_sequential(
            threshold=fixed_threshold,
            cost_scale=COST_SCALE_REF,
            mp=mp,
            mode_name=mode_name,
            seed_offset=4000000 + 10000 * pi,
        )

        penalty_pairs_frames.append(sb_pairs)
        penalty_iter_frames.append(sb_iters)

        extra = {
            "experiment": "sb_penalty_sensitivity",
            "cost_scale": COST_SCALE_REF,
            "mp": mp,
        }

        if len(sb_iters):
            extra.update({
                "runtime_sec": float(sb_iters["iteration_runtime_sec"].sum()),
                "mean_valid_cycle_rate": float(sb_iters["valid_cycle_rate"].mean()),
                "total_tabu_violations": int(sb_iters["n_tabu_violations_among_valid"].sum()),
                "mean_tabu_violation_rate_among_valid": float(sb_iters["tabu_violation_rate_among_valid"].mean(skipna=True)),
            })

        penalty_rows.append(summarize_pairs(
            sb_pairs,
            solver=mode_name,
            threshold=fixed_threshold,
            extra=extra,
        ))

penalty_sensitivity = pd.DataFrame(penalty_rows)
penalty_pairs = pd.concat(penalty_pairs_frames, ignore_index=True, sort=False) if penalty_pairs_frames else pd.DataFrame()
penalty_iterations = pd.concat(penalty_iter_frames, ignore_index=True, sort=False) if penalty_iter_frames else pd.DataFrame()


# 9. COMBINED SUMMARY AND SAVE

combined_summary = pd.concat(
    [
        threshold_sensitivity,
        scale_sensitivity,
        penalty_sensitivity,
    ],
    ignore_index=True,
    sort=False,
)

# Sort for readability
if len(combined_summary):
    sort_cols = [c for c in ["experiment", "threshold", "cost_scale", "mp", "solver"] if c in combined_summary.columns]
    combined_summary = combined_summary.sort_values(sort_cols).reset_index(drop=True)

# Save primary outputs
threshold_sensitivity.to_csv(
    SAVE_DIR / f"real_market_threshold_sensitivity_{TAG}.csv",
    index=False,
)

scale_sensitivity.to_csv(
    SAVE_DIR / f"real_market_sb_scale_sensitivity_{TAG}.csv",
    index=False,
)

penalty_sensitivity.to_csv(
    SAVE_DIR / f"real_market_sb_penalty_sensitivity_{TAG}.csv",
    index=False,
)

combined_summary.to_csv(
    SAVE_DIR / f"real_market_sensitivity_summary_{TAG}.csv",
    index=False,
)

# Save detailed accepted pairs
threshold_pairs.to_csv(
    SAVE_DIR / f"real_market_threshold_sensitivity_pairs_{TAG}.csv",
    index=False,
)

scale_pairs.to_csv(
    SAVE_DIR / f"real_market_sb_scale_sensitivity_pairs_{TAG}.csv",
    index=False,
)

penalty_pairs.to_csv(
    SAVE_DIR / f"real_market_sb_penalty_sensitivity_pairs_{TAG}.csv",
    index=False,
)

# Save SB iteration diagnostics
threshold_sb_iterations.to_csv(
    SAVE_DIR / f"real_market_threshold_sensitivity_sb_iterations_{TAG}.csv",
    index=False,
)

scale_iterations.to_csv(
    SAVE_DIR / f"real_market_sb_scale_sensitivity_iterations_{TAG}.csv",
    index=False,
)

penalty_iterations.to_csv(
    SAVE_DIR / f"real_market_sb_penalty_sensitivity_iterations_{TAG}.csv",
    index=False,
)

# Display
print("\nSaved:")
print(SAVE_DIR / f"real_market_threshold_sensitivity_{TAG}.csv")
print(SAVE_DIR / f"real_market_sb_scale_sensitivity_{TAG}.csv")
print(SAVE_DIR / f"real_market_sb_penalty_sensitivity_{TAG}.csv")
print(SAVE_DIR / f"real_market_sensitivity_summary_{TAG}.csv")

print("\nThreshold sensitivity:")
display(threshold_sensitivity)

print("\nSB cost-scale sensitivity:")
display(scale_sensitivity)

print("\nSB penalty sensitivity:")
display(penalty_sensitivity)

print("\nCombined summary:")
display(combined_summary)

SAVE_DIR: C:\Users\Антон\Desktop\курсовая\04_05
weights: 04_05\real_data_market_weights_04_05_01.csv
N stock nodes: 15
max_abs_stock_weight: 0.0269822518707688
COST_SCALE_REF: 37.06139890730726

Threshold sensitivity: -0.005

Threshold sensitivity: -0.01

Threshold sensitivity: -0.015

Threshold sensitivity: -0.02

Threshold sensitivity: -0.03

Threshold sensitivity: -0.04

Threshold sensitivity: -0.05

Threshold sensitivity: -0.055

SB cost-scale sensitivity: 1.0

SB cost-scale sensitivity: 10.0

SB cost-scale sensitivity: 25.0

SB cost-scale sensitivity: 37.06139890730726

SB cost-scale sensitivity: 50.0

SB cost-scale sensitivity: 100.0

SB penalty sensitivity: 0.5

SB penalty sensitivity: 1.0

SB penalty sensitivity: 2.0

SB penalty sensitivity: 5.0

SB penalty sensitivity: 10.0

Saved:
04_05\real_market_threshold_sensitivity_04_05_03.csv
04_05\real_market_sb_scale_sensitivity_04_05_03.csv
04_05\real_market_sb_penalty_sensitivity_04_05_03.csv
04_05\real_market_sensitivity_summary_0

,solver,threshold,n_accepted_pairs,best_path_weight,mean_path_weight,median_path_weight,n_bypass,bypass_share,duplicate_endpoint_pairs,mean_path_length_assets,max_path_length_assets,runtime_sec,experiment,cost_scale,mp,mean_valid_cycle_rate,total_tabu_violations,mean_tabu_violation_rate_among_valid
0,MILP_HiGHS_exact,-0.005,10,-0.059141,-0.056685,-0.056493,10,1.0,0,12.400000,14,3.520309,threshold_sensitivity,NaN,NaN,NaN,NaN,NaN
1,SA_valid_path,-0.005,10,-0.059141,-0.056677,-0.056474,10,1.0,0,11.800000,14,52.367409,threshold_sensitivity,NaN,NaN,NaN,NaN,NaN
2,SB_scaled_fixed_mp,-0.005,10,-0.054853,-0.051021,-0.050921,10,1.0,0,7.100000,9,21.948555,threshold_sensitivity,37.061399,1.0,0.374000,17.0,0.055901
3,MILP_HiGHS_exact,-0.010,10,-0.059141,-0.056685,-0.056493,10,1.0,0,12.400000,14,3.281091,threshold_sensitivity,NaN,NaN,NaN,NaN,NaN
4,SA_valid_path,-0.010,10,-0.059141,-0.056673,-0.056474,10,1.0,0,11.500000,13,51.534717,threshold_sensitivity,NaN,NaN,NaN,NaN,NaN
5,SB_scaled_fixed_mp,-0.010,10,-0.055148,-0.051293,-0.051846,10,1.0,0,7.500000,9,21.341940,threshold_sensitivity,37.061399,1.0,0.361000,14.0,0.047260
6,MILP_HiGHS_exact,-0.015,10,-0.059141,-0.056685,-0.056493,10,1.0,0,12.400000,14,3.204816,threshold_sensitivity,NaN,NaN,NaN,NaN,NaN
7,SA_valid_path,-0.015,10,-0.059141,-0.056677,-0.056493,10,1.0,0,11.800000,14,51.112808,threshold_sensitivity,NaN,NaN,NaN,NaN,NaN
8,SB_scaled_fixed_mp,-0.015,10,-0.055735,-0.051834,-0.051867,10,1.0,0,7.400000,9,21.268516,threshold_sensitivity,37.061399,1.0,0.392000,14.0,0.054708
9,MILP_HiGHS_exact,-0.020,10,-0.059141,-0.056685,-0.056493,10,1.0,0,12.400000,14,3.223992,threshold_sensitivity,NaN,NaN,NaN,NaN,NaN



SB cost-scale sensitivity:


,solver,threshold,n_accepted_pairs,best_path_weight,mean_path_weight,median_path_weight,n_bypass,bypass_share,duplicate_endpoint_pairs,mean_path_length_assets,max_path_length_assets,runtime_sec,experiment,cost_scale,mp,mean_valid_cycle_rate,total_tabu_violations,mean_tabu_violation_rate_among_valid
0,SB_scale_1,-0.01,0,NaN,NaN,NaN,0,NaN,0,NaN,NaN,2.136211,sb_cost_scale_sensitivity,1.000000,1.0,0.000,0,NaN
1,SB_scale_10,-0.01,0,NaN,NaN,NaN,0,NaN,0,NaN,NaN,2.140896,sb_cost_scale_sensitivity,10.000000,1.0,0.000,0,NaN
2,SB_scale_25,-0.01,10,-0.052121,-0.049443,-0.050172,10,1.0,0,6.000000,7.0,21.242200,sb_cost_scale_sensitivity,25.000000,1.0,0.231,3,0.026346
3,SB_scale_37.0614,-0.01,10,-0.053314,-0.051583,-0.052314,10,1.0,0,8.300000,11.0,21.283209,sb_cost_scale_sensitivity,37.061399,1.0,0.379,6,0.028322
4,SB_scale_50,-0.01,10,-0.055408,-0.053033,-0.053084,10,1.0,0,8.800000,10.0,21.231339,sb_cost_scale_sensitivity,50.000000,1.0,0.382,11,0.034664
5,SB_scale_100,-0.01,7,-0.059048,-0.054495,-0.054187,7,1.0,0,9.285714,11.0,16.989709,sb_cost_scale_sensitivity,100.000000,1.0,0.105,0,0.000000



SB penalty sensitivity:


,solver,threshold,n_accepted_pairs,best_path_weight,mean_path_weight,median_path_weight,n_bypass,bypass_share,duplicate_endpoint_pairs,mean_path_length_assets,max_path_length_assets,runtime_sec,experiment,cost_scale,mp,mean_valid_cycle_rate,total_tabu_violations,mean_tabu_violation_rate_among_valid
0,SB_scaled_mp_0.5,-0.01,10,-0.056255,-0.053831,-0.053868,10,1.000000,0,9.200000,11.0,21.143480,sb_penalty_sensitivity,37.061399,0.5,0.285000,6,0.026250
1,SB_scaled_mp_1,-0.01,10,-0.053808,-0.051235,-0.052243,10,1.000000,0,7.400000,8.0,21.326166,sb_penalty_sensitivity,37.061399,1.0,0.371000,16,0.066520
2,SB_scaled_mp_2,-0.01,6,-0.051146,-0.046258,-0.049659,5,0.833333,0,4.666667,6.0,14.830805,sb_penalty_sensitivity,37.061399,2.0,0.082857,1,0.142857
3,SB_scaled_mp_5,-0.01,0,NaN,NaN,NaN,0,NaN,0,NaN,NaN,2.120298,sb_penalty_sensitivity,37.061399,5.0,0.000000,0,NaN
4,SB_scaled_mp_10,-0.01,0,NaN,NaN,NaN,0,NaN,0,NaN,NaN,2.110551,sb_penalty_sensitivity,37.061399,10.0,0.000000,0,NaN



Combined summary:


,solver,threshold,n_accepted_pairs,best_path_weight,mean_path_weight,median_path_weight,n_bypass,bypass_share,duplicate_endpoint_pairs,mean_path_length_assets,max_path_length_assets,runtime_sec,experiment,cost_scale,mp,mean_valid_cycle_rate,total_tabu_violations,mean_tabu_violation_rate_among_valid
0,SB_scale_1,-0.010,0,NaN,NaN,NaN,0,NaN,0,NaN,NaN,2.136211,sb_cost_scale_sensitivity,1.000000,1.0,0.000000,0.0,NaN
1,SB_scale_10,-0.010,0,NaN,NaN,NaN,0,NaN,0,NaN,NaN,2.140896,sb_cost_scale_sensitivity,10.000000,1.0,0.000000,0.0,NaN
2,SB_scale_25,-0.010,10,-0.052121,-0.049443,-0.050172,10,1.000000,0,6.000000,7.0,21.242200,sb_cost_scale_sensitivity,25.000000,1.0,0.231000,3.0,0.026346
3,SB_scale_37.0614,-0.010,10,-0.053314,-0.051583,-0.052314,10,1.000000,0,8.300000,11.0,21.283209,sb_cost_scale_sensitivity,37.061399,1.0,0.379000,6.0,0.028322
4,SB_scale_50,-0.010,10,-0.055408,-0.053033,-0.053084,10,1.000000,0,8.800000,10.0,21.231339,sb_cost_scale_sensitivity,50.000000,1.0,0.382000,11.0,0.034664
5,SB_scale_100,-0.010,7,-0.059048,-0.054495,-0.054187,7,1.000000,0,9.285714,11.0,16.989709,sb_cost_scale_sensitivity,100.000000,1.0,0.105000,0.0,0.000000
6,SB_scaled_mp_0.5,-0.010,10,-0.056255,-0.053831,-0.053868,10,1.000000,0,9.200000,11.0,21.143480,sb_penalty_sensitivity,37.061399,0.5,0.285000,6.0,0.026250
7,SB_scaled_mp_1,-0.010,10,-0.053808,-0.051235,-0.052243,10,1.000000,0,7.400000,8.0,21.326166,sb_penalty_sensitivity,37.061399,1.0,0.371000,16.0,0.066520
8,SB_scaled_mp_2,-0.010,6,-0.051146,-0.046258,-0.049659,5,0.833333,0,4.666667,6.0,14.830805,sb_penalty_sensitivity,37.061399,2.0,0.082857,1.0,0.142857
9,SB_scaled_mp_5,-0.010,0,NaN,NaN,NaN,0,NaN,0,NaN,NaN,2.120298,sb_penalty_sensitivity,37.061399,5.0,0.000000,0.0,NaN
